# Feature Selection — CreditFlow

**Goal:** Select banking/credit-risk/DPD-relevant features from `application_train.csv`,  
then export one row per application as a JSON payload inside a CSV with columns  
`application_id` and `input_json`.  

The output CSV will be used by the simulator to replay events against the pipeline API.

**Feature groups kept:**
| Group | Why |
|---|---|
| Application identifiers | Needed as keys |
| Loan / credit amounts | Core credit-risk signals |
| Income & employment | Repayment capacity |
| DPD / delinquency (social circle) | Direct DPD proxies |
| Credit bureau enquiry counts | Hard-cut / credit-pull signals |
| Days-based features | Recency of employment, ID, registration |
| Demographics & contract type | Segmentation |
| External scores | Composite risk scores |
| TARGET | Label (kept for reference, dropped from `input_json`) |

In [ ]:
# ── Imports & Config ────────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd

RAW_SOURCE  = "data\\application_train.csv"
OUTPUT_FILE = "data\\selected_features_simulator.csv"  # application_id | input_json

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [8]:
# ── 1. Load Raw Data ─────────────────────────────────────────────────────────
org_df = pd.read_csv(RAW_SOURCE)
print(f"Shape: {org_df.shape}  |  Columns: {org_df.shape[1]}")
org_df.head(3)

Shape: (307511, 122)  |  Columns: 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0000,406597.5000,24700.5000,351000.0000,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0188,-9461,-637,-3648.0000,-2120,NaN,1,1,0,1,1,0,Laborers,1.0000,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830,0.2629,0.1394,0.0247,0.0369,0.9722,0.6192,0.0143,0.0000,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0000,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.0000,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0000,2.0000,2.0000,2.0000,-1134.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,270000.0000,1293502.5000,35698.5000,1129500.0000,Family,State servant,Higher education,Married,House / apartment,0.0035,-16765,-1188,-1186.0000,-291,NaN,1,1,0,1,1,0,Core staff,2.0000,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3113,0.6222,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.0800,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0000,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.0800,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0000,0.0000,1.0000,0.0000,-828.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,67500.0000,135000.0000,6750.0000,135000.0000,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0100,-19046,-225,-4260.0000,-2531,26.0000,1,1,1,1,1,0,Laborers,1.0000,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.5559,0.

In [9]:
# ── 2. Define Feature Groups ─────────────────────────────────────────────────

# Application identifier — used as the event key
ID_COL = "SK_ID_CURR"

# Target / label — kept for analysis reference but excluded from input_json
TARGET_COL = "TARGET"

# ── Loan / Credit Amounts ────────────────────────────────────────────────────
LOAN_AMOUNT_FEATURES = [
    "AMT_CREDIT",         # Total credit amount applied for
    "AMT_ANNUITY",        # Annuity (EMI) amount
    "AMT_GOODS_PRICE",    # Price of goods if loan is consumer credit
    "AMT_INCOME_TOTAL",   # Total annual income
]

# ── DPD / Delinquency Signals ────────────────────────────────────────────────
# OBS = observations, DEF = defaults (DPD events) in social circle
DPD_FEATURES = [
    "OBS_30_CNT_SOCIAL_CIRCLE",   # Observations with DPD30+ in applicant's social circle
    "DEF_30_CNT_SOCIAL_CIRCLE",   # Defaults with DPD30+ in social circle
    "OBS_60_CNT_SOCIAL_CIRCLE",   # Observations with DPD60+ in social circle
    "DEF_60_CNT_SOCIAL_CIRCLE",   # Defaults with DPD60+ in social circle (hard-cut signal)
]

# ── Credit Bureau Enquiry Counts (Hard-Cut / Soft-Pull indicators) ────────────
CREDIT_BUREAU_FEATURES = [
    "AMT_REQ_CREDIT_BUREAU_HOUR",  # Credit enquiries in last hour
    "AMT_REQ_CREDIT_BUREAU_DAY",   # Credit enquiries in last day
    "AMT_REQ_CREDIT_BUREAU_WEEK",  # Credit enquiries in last week
    "AMT_REQ_CREDIT_BUREAU_MON",   # Credit enquiries in last month
    "AMT_REQ_CREDIT_BUREAU_QRT",   # Credit enquiries in last quarter
    "AMT_REQ_CREDIT_BUREAU_YEAR",  # Credit enquiries in last year
]

# ── External Risk Scores ─────────────────────────────────────────────────────
EXTERNAL_SCORE_FEATURES = [
    "EXT_SOURCE_1",   # External score 1 (bureau / scoring agency)
    "EXT_SOURCE_2",   # External score 2
    "EXT_SOURCE_3",   # External score 3
]

# ── Days-Based Features (recency / tenure) ───────────────────────────────────
DAYS_FEATURES = [
    "DAYS_BIRTH",           # Age in days (negative: days before application)
    "DAYS_EMPLOYED",        # Employment start relative to application
    "DAYS_REGISTRATION",    # Days since registration document changed
    "DAYS_ID_PUBLISH",      # Days since ID was changed/published
    "DAYS_LAST_PHONE_CHANGE",  # Days since last phone change
]

# ── Demographics & Application Metadata ─────────────────────────────────────
DEMOGRAPHIC_FEATURES = [
    "NAME_CONTRACT_TYPE",    # Cash loan vs Revolving
    "CODE_GENDER",           # Gender
    "FLAG_OWN_CAR",          # Owns a car
    "FLAG_OWN_REALTY",       # Owns real estate
    "CNT_CHILDREN",          # Number of children
    "CNT_FAM_MEMBERS",       # Family size
    "NAME_INCOME_TYPE",      # Employment category (Salaried, Self-employed, etc.)
    "NAME_EDUCATION_TYPE",   # Highest education level
    "NAME_FAMILY_STATUS",    # Marital status
    "NAME_HOUSING_TYPE",     # Type of housing
    "OCCUPATION_TYPE",       # Occupation
    "ORGANIZATION_TYPE",     # Employer organization type
    "REGION_RATING_CLIENT",  # Region risk rating (1-3)
    "REGION_RATING_CLIENT_W_CITY",  # Region risk with city
]

# ── Combine all selected features ───────────────────────────────────────────
SELECTED_FEATURES = (
    LOAN_AMOUNT_FEATURES
    + DPD_FEATURES
    + CREDIT_BUREAU_FEATURES
    + EXTERNAL_SCORE_FEATURES
    + DAYS_FEATURES
    + DEMOGRAPHIC_FEATURES
)

print(f"Total selected features (excluding ID & TARGET): {len(SELECTED_FEATURES)}")
print("\nFeature list:")
for f in SELECTED_FEATURES:
    print(f"  {f}")

Total selected features (excluding ID & TARGET): 36

Feature list:
  AMT_CREDIT
  AMT_ANNUITY
  AMT_GOODS_PRICE
  AMT_INCOME_TOTAL
  OBS_30_CNT_SOCIAL_CIRCLE
  DEF_30_CNT_SOCIAL_CIRCLE
  OBS_60_CNT_SOCIAL_CIRCLE
  DEF_60_CNT_SOCIAL_CIRCLE
  AMT_REQ_CREDIT_BUREAU_HOUR
  AMT_REQ_CREDIT_BUREAU_DAY
  AMT_REQ_CREDIT_BUREAU_WEEK
  AMT_REQ_CREDIT_BUREAU_MON
  AMT_REQ_CREDIT_BUREAU_QRT
  AMT_REQ_CREDIT_BUREAU_YEAR
  EXT_SOURCE_1
  EXT_SOURCE_2
  EXT_SOURCE_3
  DAYS_BIRTH
  DAYS_EMPLOYED
  DAYS_REGISTRATION
  DAYS_ID_PUBLISH
  DAYS_LAST_PHONE_CHANGE
  NAME_CONTRACT_TYPE
  CODE_GENDER
  FLAG_OWN_CAR
  FLAG_OWN_REALTY
  CNT_CHILDREN
  CNT_FAM_MEMBERS
  NAME_INCOME_TYPE
  NAME_EDUCATION_TYPE
  NAME_FAMILY_STATUS
  NAME_HOUSING_TYPE
  OCCUPATION_TYPE
  ORGANIZATION_TYPE
  REGION_RATING_CLIENT
  REGION_RATING_CLIENT_W_CITY


In [10]:
# ── 3. Null / Missing Value Analysis on Selected Features ────────────────────
subset = org_df[[ID_COL, TARGET_COL] + SELECTED_FEATURES].copy()

null_summary = pd.DataFrame({
    "null_count": subset[SELECTED_FEATURES].isnull().sum(),
    "null_pct":   (subset[SELECTED_FEATURES].isnull().sum() / len(subset) * 100).round(2),
    "dtype":      subset[SELECTED_FEATURES].dtypes,
}).sort_values("null_pct", ascending=False)

print(f"Dataset rows: {len(subset):,}")
print("\nNull analysis on selected features:")
null_summary

Dataset rows: 307,511

Null analysis on selected features:


,null_count,null_pct,dtype
EXT_SOURCE_1,173378,56.3800,float64
OCCUPATION_TYPE,96391,31.3500,str
EXT_SOURCE_3,60965,19.8300,float64
AMT_REQ_CREDIT_BUREAU_MON,41519,13.5000,float64
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.5000,float64
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.5000,float64
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.5000,float64
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.5000,float64
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.5000,float64
OBS_30_CNT_SOCIAL_CIRCLE,1021,0.3300,float64


In [11]:
# ── 4. Basic Statistics ───────────────────────────────────────────────────────
print("=== Numeric features ===")
numeric_features = [
    f for f in SELECTED_FEATURES
    if subset[f].dtype in ["int64", "float64"]
]
subset[numeric_features].describe().T

=== Numeric features ===


,count,mean,std,min,25%,50%,75%,max
AMT_CREDIT,307511.0000,599025.9997,402490.7770,45000.0000,270000.0000,513531.0000,808650.0000,4050000.0000
AMT_ANNUITY,307499.0000,27108.5739,14493.7373,1615.5000,16524.0000,24903.0000,34596.0000,258025.5000
AMT_GOODS_PRICE,307233.0000,538396.2074,369446.4605,40500.0000,238500.0000,450000.0000,679500.0000,4050000.0000
AMT_INCOME_TOTAL,307511.0000,168797.9193,237123.1463,25650.0000,112500.0000,147150.0000,202500.0000,117000000.0000
OBS_30_CNT_SOCIAL_CIRCLE,306490.0000,1.4222,2.4010,0.0000,0.0000,0.0000,2.0000,348.0000
DEF_30_CNT_SOCIAL_CIRCLE,306490.0000,0.1434,0.4467,0.0000,0.0000,0.0000,0.0000,34.0000
OBS_60_CNT_SOCIAL_CIRCLE,306490.0000,1.4053,2.3798,0.0000,0.0000,0.0000,2.0000,344.0000
DEF_60_CNT_SOCIAL_CIRCLE,306490.0000,0.1000,0.3623,0.0000,0.0000,0.0000,0.0000,24.0000
AMT_REQ_CREDIT_BUREAU_HOUR,265992.0000,0.0064,0.0838,0.0000,0.0000,0.0000,0.0000,4.0000
AMT_REQ_CREDIT_BUREAU_DAY,265992.0000,0.0070,0.1108,0.0000,0.0000,0.0000,0.0000,9.0000


In [12]:
# ── 5. Categorical Feature Value Counts ──────────────────────────────────────
cat_features = [
    f for f in SELECTED_FEATURES
    if subset[f].dtype == "object"
]
print(f"Categorical features: {cat_features}\n")
for col in cat_features:
    print(f"\n── {col} ──")
    print(subset[col].value_counts(dropna=False).head(10))

Categorical features: []



In [13]:
# ── 6. DPD / Delinquency Signal Analysis ─────────────────────────────────────
print("DPD 30-day defaults in social circle:")
print(subset["DEF_30_CNT_SOCIAL_CIRCLE"].value_counts(dropna=False).head())

print("\nDPD 60-day defaults in social circle:")
print(subset["DEF_60_CNT_SOCIAL_CIRCLE"].value_counts(dropna=False).head())

print("\nCredit bureau enquiries (last year):")
print(subset["AMT_REQ_CREDIT_BUREAU_YEAR"].value_counts(dropna=False).head(10))

print("\nCredit bureau enquiries (last hour) — hard-cut signal:")
print(subset["AMT_REQ_CREDIT_BUREAU_HOUR"].value_counts(dropna=False).head(10))

DPD 30-day defaults in social circle:
DEF_30_CNT_SOCIAL_CIRCLE
0.0000    271324
1.0000     28328
2.0000      5323
3.0000      1192
NaN         1021
Name: count, dtype: int64

DPD 60-day defaults in social circle:
DEF_60_CNT_SOCIAL_CIRCLE
0.0000    280721
1.0000     21841
2.0000      3170
NaN         1021
3.0000       598
Name: count, dtype: int64

Credit bureau enquiries (last year):
AMT_REQ_CREDIT_BUREAU_YEAR
0.0000    71801
1.0000    63405
2.0000    50192
NaN       41519
3.0000    33628
4.0000    20714
5.0000    12052
6.0000     6967
7.0000     3869
8.0000     2127
Name: count, dtype: int64

Credit bureau enquiries (last hour) — hard-cut signal:
AMT_REQ_CREDIT_BUREAU_HOUR
0.0000    264366
NaN        41519
1.0000      1560
2.0000        56
3.0000         9
4.0000         1
Name: count, dtype: int64


In [14]:
# ── 7. Derived / Engineered Features for Analytics ───────────────────────────
# These are lightweight derived columns added to the payload to make
# PySpark analytics simpler downstream.

# Debt-to-Income ratio
subset["DEBT_TO_INCOME"] = (
    subset["AMT_CREDIT"] / subset["AMT_INCOME_TOTAL"].replace(0, np.nan)
).round(4)

# Annuity-to-Income ratio (EMI burden)
subset["ANNUITY_TO_INCOME"] = (
    subset["AMT_ANNUITY"] / subset["AMT_INCOME_TOTAL"].replace(0, np.nan)
).round(4)

# Loan-to-Value ratio (credit vs goods price)
subset["LOAN_TO_VALUE"] = (
    subset["AMT_CREDIT"] / subset["AMT_GOODS_PRICE"].replace(0, np.nan)
).round(4)

# Age in years (absolute value: DAYS_BIRTH is negative)
subset["AGE_YEARS"] = (subset["DAYS_BIRTH"].abs() / 365).round(1)

# Years employed (DAYS_EMPLOYED=365243 means unemployed/pensioner — cap it)
subset["YEARS_EMPLOYED"] = (
    subset["DAYS_EMPLOYED"]
    .where(subset["DAYS_EMPLOYED"] < 0, other=np.nan)  # 365243 = anomaly
    .abs() / 365
).round(1)

# Total DPD exposure (social circle)
subset["TOTAL_DPD_SOCIAL_CIRCLE"] = (
    subset["DEF_30_CNT_SOCIAL_CIRCLE"].fillna(0)
    + subset["DEF_60_CNT_SOCIAL_CIRCLE"].fillna(0)
)

# Total credit bureau hits (all windows)
bureau_cols = CREDIT_BUREAU_FEATURES
subset["TOTAL_BUREAU_ENQUIRIES"] = subset[bureau_cols].fillna(0).sum(axis=1)

DERIVED_FEATURES = [
    "DEBT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "LOAN_TO_VALUE",
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "TOTAL_DPD_SOCIAL_CIRCLE",
    "TOTAL_BUREAU_ENQUIRIES",
]

print(f"Derived features added: {DERIVED_FEATURES}")
subset[DERIVED_FEATURES].describe().T

Derived features added: ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'LOAN_TO_VALUE', 'AGE_YEARS', 'YEARS_EMPLOYED', 'TOTAL_DPD_SOCIAL_CIRCLE', 'TOTAL_BUREAU_ENQUIRIES']


,count,mean,std,min,25%,50%,75%,max
DEBT_TO_INCOME,307511.0000,3.9576,2.6897,0.0048,2.0187,3.2651,5.1599,84.7368
ANNUITY_TO_INCOME,307499.0000,0.1809,0.0946,0.0002,0.1148,0.1628,0.2291,1.8760
LOAN_TO_VALUE,307233.0000,1.1230,0.1240,0.1500,1.0000,1.1188,1.1980,6.0000
AGE_YEARS,307511.0000,43.9370,11.9561,20.5000,34.0000,43.2000,53.9000,69.1000
YEARS_EMPLOYED,252135.0000,6.5320,6.4066,0.0000,2.1000,4.5000,8.7000,49.1000
TOTAL_DPD_SOCIAL_CIRCLE,307511.0000,0.2427,0.7794,0.0000,0.0000,0.0000,0.0000,58.0000
TOTAL_BUREAU_ENQUIRIES,307511.0000,2.1457,2.2909,0.0000,0.0000,2.0000,3.0000,262.0000


In [15]:
# ── 8. Final Feature Set ──────────────────────────────────────────────────────
FINAL_PAYLOAD_FEATURES = SELECTED_FEATURES + DERIVED_FEATURES

print(f"Total payload features: {len(FINAL_PAYLOAD_FEATURES)}")
print(f"  ● Original selected : {len(SELECTED_FEATURES)}")
print(f"  ● Derived           : {len(DERIVED_FEATURES)}")
print("\nFinal payload columns:")
for f in FINAL_PAYLOAD_FEATURES:
    print(f"  {f}")

Total payload features: 43
  ● Original selected : 36
  ● Derived           : 7

Final payload columns:
  AMT_CREDIT
  AMT_ANNUITY
  AMT_GOODS_PRICE
  AMT_INCOME_TOTAL
  OBS_30_CNT_SOCIAL_CIRCLE
  DEF_30_CNT_SOCIAL_CIRCLE
  OBS_60_CNT_SOCIAL_CIRCLE
  DEF_60_CNT_SOCIAL_CIRCLE
  AMT_REQ_CREDIT_BUREAU_HOUR
  AMT_REQ_CREDIT_BUREAU_DAY
  AMT_REQ_CREDIT_BUREAU_WEEK
  AMT_REQ_CREDIT_BUREAU_MON
  AMT_REQ_CREDIT_BUREAU_QRT
  AMT_REQ_CREDIT_BUREAU_YEAR
  EXT_SOURCE_1
  EXT_SOURCE_2
  EXT_SOURCE_3
  DAYS_BIRTH
  DAYS_EMPLOYED
  DAYS_REGISTRATION
  DAYS_ID_PUBLISH
  DAYS_LAST_PHONE_CHANGE
  NAME_CONTRACT_TYPE
  CODE_GENDER
  FLAG_OWN_CAR
  FLAG_OWN_REALTY
  CNT_CHILDREN
  CNT_FAM_MEMBERS
  NAME_INCOME_TYPE
  NAME_EDUCATION_TYPE
  NAME_FAMILY_STATUS
  NAME_HOUSING_TYPE
  OCCUPATION_TYPE
  ORGANIZATION_TYPE
  REGION_RATING_CLIENT
  REGION_RATING_CLIENT_W_CITY
  DEBT_TO_INCOME
  ANNUITY_TO_INCOME
  LOAN_TO_VALUE
  AGE_YEARS
  YEARS_EMPLOYED
  TOTAL_DPD_SOCIAL_CIRCLE
  TOTAL_BUREAU_ENQUIRIES


In [16]:
# ── 9. Build Simulator CSV  (application_id | input_json) ────────────────────
#
# Format
#   application_id  → "APP-<SK_ID_CURR>"  (matches README event envelope)
#   input_json      → JSON string with all selected + derived features
#                     TARGET is intentionally excluded from input_json
#                     (it's a label, not an input field)
#
# NaN handling: convert to None so JSON serialises cleanly as null

def row_to_json(row: pd.Series) -> str:
    """Serialise a single row to a compact JSON string; NaN → null."""
    payload = {}
    for col in FINAL_PAYLOAD_FEATURES:
        val = row[col]
        if pd.isna(val):
            payload[col] = None
        elif isinstance(val, (np.integer,)):
            payload[col] = int(val)
        elif isinstance(val, (np.floating,)):
            payload[col] = float(val)
        else:
            payload[col] = val
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


print("Building simulator rows...")
simulator_rows = []
for _, row in subset.iterrows():
    simulator_rows.append({
        "application_id": f"APP-{int(row[ID_COL])}",
        "input_json":     row_to_json(row),
    })

sim_df = pd.DataFrame(simulator_rows, columns=["application_id", "input_json"])
print(f"Simulator DataFrame shape: {sim_df.shape}")
sim_df.head(2)

Building simulator rows...
Simulator DataFrame shape: (307511, 2)


,application_id,input_json
0,APP-100002,"{""AMT_CREDIT"":406597.5,""AMT_ANNUITY"":24700.5,""..."
1,APP-100003,"{""AMT_CREDIT"":1293502.5,""AMT_ANNUITY"":35698.5,..."


In [17]:
# ── 10. Verify — peek at one JSON payload ────────────────────────────────────
sample_json = json.loads(sim_df.iloc[0]["input_json"])
print(f"application_id : {sim_df.iloc[0]['application_id']}")
print(f"Payload keys   : {len(sample_json)}")
print("\nSample payload (pretty):")
print(json.dumps(sample_json, indent=2))

application_id : APP-100002
Payload keys   : 43

Sample payload (pretty):
{
  "AMT_CREDIT": 406597.5,
  "AMT_ANNUITY": 24700.5,
  "AMT_GOODS_PRICE": 351000.0,
  "AMT_INCOME_TOTAL": 202500.0,
  "OBS_30_CNT_SOCIAL_CIRCLE": 2.0,
  "DEF_30_CNT_SOCIAL_CIRCLE": 2.0,
  "OBS_60_CNT_SOCIAL_CIRCLE": 2.0,
  "DEF_60_CNT_SOCIAL_CIRCLE": 2.0,
  "AMT_REQ_CREDIT_BUREAU_HOUR": 0.0,
  "AMT_REQ_CREDIT_BUREAU_DAY": 0.0,
  "AMT_REQ_CREDIT_BUREAU_WEEK": 0.0,
  "AMT_REQ_CREDIT_BUREAU_MON": 0.0,
  "AMT_REQ_CREDIT_BUREAU_QRT": 0.0,
  "AMT_REQ_CREDIT_BUREAU_YEAR": 1.0,
  "EXT_SOURCE_1": 0.08303696739132256,
  "EXT_SOURCE_2": 0.2629485927471776,
  "EXT_SOURCE_3": 0.13937578009978951,
  "DAYS_BIRTH": -9461,
  "DAYS_EMPLOYED": -637,
  "DAYS_REGISTRATION": -3648.0,
  "DAYS_ID_PUBLISH": -2120,
  "DAYS_LAST_PHONE_CHANGE": -1134.0,
  "NAME_CONTRACT_TYPE": "Cash loans",
  "CODE_GENDER": "M",
  "FLAG_OWN_CAR": "N",
  "FLAG_OWN_REALTY": "Y",
  "CNT_CHILDREN": 0,
  "CNT_FAM_MEMBERS": 1.0,
  "NAME_INCOME_TYPE": "Working",


In [18]:
# ── 11. Null Coverage Check on Final Payload ─────────────────────────────────
# Expand input_json back to DataFrame to check null rates in payload
expanded = pd.json_normalize(sim_df["input_json"].apply(json.loads))
null_in_payload = (
    expanded.isnull().sum() / len(expanded) * 100
).round(2).sort_values(ascending=False)

print("Columns with nulls in payload (% null):")
print(null_in_payload[null_in_payload > 0].to_string())

Columns with nulls in payload (% null):
EXT_SOURCE_1                 56.3800
OCCUPATION_TYPE              31.3500
EXT_SOURCE_3                 19.8300
YEARS_EMPLOYED               18.0100
AMT_REQ_CREDIT_BUREAU_QRT    13.5000
AMT_REQ_CREDIT_BUREAU_MON    13.5000
AMT_REQ_CREDIT_BUREAU_YEAR   13.5000
AMT_REQ_CREDIT_BUREAU_WEEK   13.5000
AMT_REQ_CREDIT_BUREAU_HOUR   13.5000
AMT_REQ_CREDIT_BUREAU_DAY    13.5000
DEF_30_CNT_SOCIAL_CIRCLE      0.3300
OBS_30_CNT_SOCIAL_CIRCLE      0.3300
DEF_60_CNT_SOCIAL_CIRCLE      0.3300
OBS_60_CNT_SOCIAL_CIRCLE      0.3300
EXT_SOURCE_2                  0.2100
LOAN_TO_VALUE                 0.0900
AMT_GOODS_PRICE               0.0900


In [19]:
# ── 12. Save Output CSV ───────────────────────────────────────────────────────
# QUOTE_ALL ensures JSON strings with embedded commas/newlines stay safe.
import csv as _csv
sim_df.to_csv(OUTPUT_FILE, index=False, quoting=_csv.QUOTE_ALL)
print(f"✅ Saved → {OUTPUT_FILE}")
print(f"   Rows    : {len(sim_df):,}")
print(f"   Columns : {list(sim_df.columns)}")

# Quick sanity read-back
check = pd.read_csv(OUTPUT_FILE)
print(f"\nRead-back shape : {check.shape}")
bad_rows = check[~check["application_id"].str.startswith("APP-", na=False)]
print(f"Rows with invalid app_id: {len(bad_rows)}  (expected 0)")
check.head(2)

## Summary

| Item | Value |
|---|---|
| Source file | `application_train.csv` |
| Output file | `selected_features_simulator.csv` |
| Output columns | `application_id`, `input_json` |
| Rows | Same as source |
| Original features selected | 28 |
| Derived features | 7 |
| Total payload features | 35 |

### How to use in the Simulator

```python
import pandas as pd, json, requests

sim_df = pd.read_csv("selected_features_simulator.csv")

API_URL = "http://localhost:8000/api/v1/application"

for _, row in sim_df.iterrows():
    payload = json.loads(row["input_json"])
    payload["application_id"] = row["application_id"]
    response = requests.post(API_URL, json=payload)
    print(row["application_id"], response.status_code)
```

### Feature Groups in Payload

| Group | Features |
|---|---|
| Loan/Credit amounts | AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE, AMT_INCOME_TOTAL |
| DPD (social circle) | OBS_30/60_CNT_SOCIAL_CIRCLE, DEF_30/60_CNT_SOCIAL_CIRCLE |
| Credit bureau enquiries | AMT_REQ_CREDIT_BUREAU_HOUR/DAY/WEEK/MON/QRT/YEAR |
| External risk scores | EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3 |
| Days-based recency | DAYS_BIRTH, DAYS_EMPLOYED, DAYS_REGISTRATION, DAYS_ID_PUBLISH, DAYS_LAST_PHONE_CHANGE |
| Demographics | NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR/REALTY, CNT_CHILDREN, CNT_FAM_MEMBERS, NAME_INCOME/EDUCATION/FAMILY/HOUSING_TYPE, OCCUPATION_TYPE, ORGANIZATION_TYPE, REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY |
| Derived | DEBT_TO_INCOME, ANNUITY_TO_INCOME, LOAN_TO_VALUE, AGE_YEARS, YEARS_EMPLOYED, TOTAL_DPD_SOCIAL_CIRCLE, TOTAL_BUREAU_ENQUIRIES |